# EDA — Profiles (KMeans) with Named Segments

**Profiles (fixed mapping)**  
- Cluster 0 → **Young & Students**  
- Cluster 1 → **Retired / Stable**  
- Cluster 2 → **Middle-age with Loans**  
- Cluster 3 → **High-balance Professionals**

**What**: Build 4 clusters with KMeans, attach human-readable names, summarize, and explore variables.  
**Why**: Quickly reason about segments (size, yes-rate, medians, dominant categories).  
**How**: Impute/scale numeric + one-hot categorical → KMeans(4) → name clusters → summary → Plotly charts.


In [ ]:
# What : Load the Bank Marketing dataset
# Why  : All EDA/segmentation needs a clean DataFrame
# How  : Try common paths; read with pandas

import os, time, pathlib
import numpy as np, pandas as pd
import plotly.express as px
from IPython.display import display

CANDIDATES = [
    os.getenv("BANK_DATA_PATH"),
    "/home/ekabwe/code/tsonma/lewagon_cust_targeting/data/bank-full.csv",
    "data/bank-full.csv", "../data/bank-full.csv", "bank-full.csv"
]
DATA_PATH = None
for p in CANDIDATES:
    if not p:
        continue
    fp = pathlib.Path(p).expanduser()
    if fp.exists():
        DATA_PATH = fp
        break
if DATA_PATH is None:
    raise FileNotFoundError("bank-full.csv not found. Set BANK_DATA_PATH or place file under ./data/.")

df = pd.read_csv(DATA_PATH, sep=";")
assert "y" in df.columns, "Expected target column 'y' (values 'yes'/'no')."

display(df.head())
print(f"Loaded: {DATA_PATH} — {df.shape[0]:,} rows × {df.shape[1]} cols")


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


Loaded: /home/ekabwe/code/tsonma/lewagon_cust_targeting/data/bank-full.csv — 45,211 rows × 17 cols


In [14]:
# What : Preprocess (num/cat) and fit KMeans with 4 clusters
# Why  : Create unsupervised customer profiles
# How  : SimpleImputer+StandardScaler for numeric, OneHot for categoricals → KMeans

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans

X = df.drop(columns=["y"]).copy()
num_features = X.select_dtypes(include=[np.number]).columns.tolist()
cat_features = [c for c in X.columns if c not in num_features]

num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler())
])

try:
    cat_ohe = OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse_output=False)
except TypeError:
    cat_ohe = OneHotEncoder(handle_unknown="ignore", drop="if_binary", sparse=False)

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("ohe", cat_ohe)
])

pre = ColumnTransformer(
    [("num", num_pipe, num_features),
     ("cat", cat_pipe, cat_features)],
    remainder="drop"
)

kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
pipe = Pipeline([("pre", pre), ("km", kmeans)])

t0 = time.time()
pipe.fit(X)
elapsed = time.time() - t0

dfp = df.copy()
dfp["cluster"] = pipe.named_steps["km"].labels_.astype(int)
dfp["y_bin"] = dfp["y"].map({"yes":1,"no":0})

print(f"KMeans fitted in {elapsed:.2f}s | {len(num_features)} numeric, {len(cat_features)} categorical")
dfp[["cluster","y"]].head()


KMeans fitted in 3.04s | 7 numeric, 9 categorical


,cluster,y
0,0,no
1,1,no
2,1,no
3,0,no
4,1,no


In [ ]:
# What : Map cluster IDs to human-readable profile names
# Why  : Communicate insights clearly to stakeholders
# How  : Fixed mapping (0..3) → names provided

PROFILE_MAP = {
    0: "Young & Students",
    1: "Retired / Stable",
    2: "Middle-age with Loans",
    3: "High-balance Professionals",
}

dfp["profile"] = dfp["cluster"].map(PROFILE_MAP)
dfp["profile"] = pd.Categorical(
    dfp["profile"],
    categories=[PROFILE_MAP[i] for i in sorted(PROFILE_MAP.keys())],
    ordered=True
)

dfp[["cluster","profile"]].head()


,cluster,profile
0,0,Young & Students
1,1,Retired / Stable
2,1,Retired / Stable
3,0,Young & Students
4,1,Retired / Stable


In [16]:
# What : Produce quick summaries by profile
# Why  : Size, yes-rate, medians help to characterize segments
# How  : groupby + agg; top categories helper

def top_cats(df_in, col, topk=2):
    if col not in df_in.columns or pd.api.types.is_numeric_dtype(df_in[col]):
        return None
    tab = (pd.crosstab(df_in["profile"], df_in[col], normalize="index")
             .stack().rename("p").reset_index())
    out = {}
    for prof in df_in["profile"].cat.categories:
        sub = tab[tab["profile"] == prof].sort_values("p", ascending=False).head(topk)
        out[prof] = ", ".join(f"{r[col]} ({r['p']:.0%})" for _, r in sub.iterrows())
    return out

summary = (
    dfp.groupby("profile")
       .agg(size=("y_bin","size"),
            yes_rate=("y_bin","mean"),
            age_median=("age","median") if "age" in dfp else ("y_bin","mean"),
            balance_median=("balance","median") if "balance" in dfp else ("y_bin","mean"))
       .round({"yes_rate":3, "age_median":1, "balance_median":0})
       .reset_index()
)

top_job = top_cats(dfp, "job", 2)
top_mar = top_cats(dfp, "marital", 2)
top_pout = top_cats(dfp, "poutcome", 1)

display(summary)

# Quick descriptor table
rows = []
for prof in dfp["profile"].cat.categories:
    r = summary.loc[summary["profile"] == prof].iloc[0]
    desc_parts = []
    if top_job and prof in top_job: desc_parts.append(f"Jobs: {top_job[prof]}")
    if top_mar and prof in top_mar: desc_parts.append(f"Marital: {top_mar[prof]}")
    if top_pout and prof in top_pout: desc_parts.append(f"Poutcome: {top_pout[prof]}")
    rows.append({
        "Profile": prof,
        "Size": int(r["size"]),
        "Yes rate": f"{r['yes_rate']*100:.1f}%",
        "Median age": r.get("age_median", np.nan),
        "Median balance": r.get("balance_median", np.nan),
        "Description": " / ".join(desc_parts)
    })

pd.DataFrame(rows)


/tmp/ipykernel_173069/2476688990.py:17: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,profile,size,yes_rate,age_median,balance_median
0,Young & Students,14552,0.105,51.0,642.0
1,Retired / Stable,21871,0.104,34.0,342.0
2,Middle-age with Loans,7002,0.202,38.0,556.0
3,High-balance Professionals,1786,0.038,39.0,304.0


,Profile,Size,Yes rate,Median age,Median balance,Description
0,Young & Students,14552,10.5%,51.0,642.0,"Jobs: blue-collar (20%), management (20%) / Ma..."
1,Retired / Stable,21871,10.4%,34.0,342.0,"Jobs: blue-collar (22%), management (21%) / Ma..."
2,Middle-age with Loans,7002,20.2%,38.0,556.0,"Jobs: blue-collar (22%), management (22%) / Ma..."
3,High-balance Professionals,1786,3.8%,39.0,304.0,"Jobs: management (24%), blue-collar (21%) / Ma..."


## Visualize a chosen variable (overall or by profile)

- **Numeric**
  - Histogram (% grouped by `y`)
  - Boxplot for skewed variables: `balance`, `duration`, `campaign`, `pdays`, `previous`

- **Categorical**
  - Stacked bar charts (proportion by category)


In [26]:
# What : Interactive explorer with dropdowns for Profile & Filter
# Why  : Compare distributions overall vs within a chosen customer profile (cluster)
# How  : ipywidgets (Dropdown) + Plotly (hist/box/stacked) with automatic logic

import pandas as pd
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

# --- Safety: pick the dataframe you use in this notebook ---
# Use dfp if you created it with profiles; otherwise fall back to df
data_ = dfp.copy() if 'dfp' in globals() else df.copy()
assert 'y' in data_.columns, "Expected target column 'y'."
assert 'profile' in data_.columns, "Expected 'profile' column (create it from KMeans first)."

# --- Dropdown options ---
profile_order = (list(data_['profile'].cat.categories)
                 if pd.api.types.is_categorical_dtype(data_['profile'])
                 else sorted(data_['profile'].unique(), key=lambda x: str(x)))
profiles = ['All'] + profile_order

exclude_cols = {'y', 'y_bin', 'cluster', 'profile'}
variables = [c for c in data_.columns if c not in exclude_cols]

# --- Plot function ---
def _visualize(var, profile):
    color_map = {"yes":"#2ca02c", "no":"#ff7f0e"}
    numeric_skewed = {"balance","duration","campaign","pdays","previous"}

    dplot = data_ if profile == 'All' else data_[data_['profile'] == profile]
    if dplot.empty:
        print("No data for this selection."); return

    is_numeric = pd.api.types.is_numeric_dtype(dplot[var])

    if is_numeric:
        if var in numeric_skewed:
            fig = px.box(
                dplot, x="y", y=var, color="y",
                color_discrete_map=color_map,
                title=f"{var} by target — {profile}"
            )
        else:
            fig = px.histogram(
                dplot, x=var, color="y",
                nbins=30, histnorm="percent", barmode="group",
                color_discrete_map=color_map,
                title=f"{var} vs target — {profile}"
            )
    else:
        fig = px.histogram(
            dplot, x=var, color="y",
            barmode="stack", barnorm="percent",
            color_discrete_map=color_map,
            title=f"{var} vs target — {profile}"
        )

    fig.update_layout(legend_title_text="Subscription", legend=dict(traceorder="reversed"))
    fig.show()

# --- Widgets ---
w_profile = widgets.Dropdown(options=profiles, value='All', description='Profile')
w_filter  = widgets.Dropdown(options=variables, value=variables[0], description='Filter by')

ui = widgets.HBox([w_profile, w_filter])
out = widgets.interactive_output(
    lambda Profile, Variable: _visualize(Variable, Profile),
    {'Profile': w_profile, 'Variable': w_filter}
)

display(ui, out)


/tmp/ipykernel_173069/3561333008.py:18: DeprecationWarning:

is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead



Output()